# MedNorm E5 XLM-R MRC-NER Training

Status: READY_FOR_COLAB_SMOKE. Intended environment: Colab GPU with Drive mounted. Expected artifact directories:

- `/content/drive/MyDrive/MedNorm-VI/artifacts/e5_xlmr_mrc_smoke_v1`
- `/content/drive/MyDrive/MedNorm-VI/artifacts/e5_xlmr_mrc_full_v1`

Full training requires `I_AUTHORIZE_E5_FULL_TRAINING`. The five organizer queries are versioned by `mrc-type-queries-v1` and their hash is written into every full artifact. Query tokens are masked from labels, context offsets are validated before model acquisition, and no cell runs organizer inference or writes `output.zip`.

Set `RUN_SMOKE_TRAINING=True` for the bounded smoke path, then keep it false and set `RUN_FULL_TRAINING=True` with the full authorization string for full training.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import re
import subprocess

from mednorm_vi.training.phase2.e5_mrc_training import (
    E5_FULL_AUTHORIZATION,
    assert_full_not_initialized_from_smoke,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
CORPUS_DIR = DRIVE_ROOT / "data" / "processed"
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e5_xlmr_mrc_smoke_v1"
FULL_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e5_xlmr_mrc_full_v1"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
RUN_FULL_TRAINING = False
RUN_SMOKE_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
RESUME_FROM_FULL_CHECKPOINT = False
SEED = 20260727
SMOKE_EPOCHS = 1
FULL_EPOCHS = 12
EFFECTIVE_BATCH_SIZE = 8
PINNED_MODEL_REVISION = os.environ.get("MEDNORM_E5_MODEL_REVISION", "")
PINNED_TOKENIZER_REVISION = os.environ.get("MEDNORM_E5_TOKENIZER_REVISION", "")
OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
random.seed(SEED)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)

assert_full_not_initialized_from_smoke(
    run_full_training=RUN_FULL_TRAINING,
    resume_from_smoke_checkpoint=RESUME_FROM_SMOKE_CHECKPOINT,
)
if RUN_FULL_TRAINING and CONFIRM_FULL != E5_FULL_AUTHORIZATION:
    raise SystemExit("E5 full training requires explicit operator authorization")


In [ ]:
EXPECTED_CORPUS_HASHES = {
    "public_ner_train.jsonl": "892dc22d7e051e05f9c96d90f42dfde7f38083a74bba6fe65b5c1d9dd05e2a4a",
    "public_ner_validation.jsonl": "ed7cdd2d49799cef0a868b6c75a3df4ca1e93ed03223337a7d31afe40f68f103",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_corpus_hashes(CORPUS_DIR: Path) -> dict[str, str]:
    observed = {}
    for name, expected in EXPECTED_CORPUS_HASHES.items():
        path = CORPUS_DIR / name
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = sha256_file(path)
        if digest != expected:
            raise AssertionError(f"corpus hash mismatch for {name}")
        observed[name] = digest
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)

RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", RESOLVED_COMMIT):
    raise AssertionError("repository commit must be a 40-hex SHA")

def require_resolved_revision(value: str, field_name: str) -> None:
    if not re.fullmatch(r"[0-9a-f]{40}", value):
        raise SystemExit(f"{field_name} must be resolved to an immutable 40-hex revision in the bootstrap cell")

if RUN_FULL_TRAINING or RUN_SMOKE_TRAINING:
    require_resolved_revision(PINNED_MODEL_REVISION, "PINNED_MODEL_REVISION")
    require_resolved_revision(PINNED_TOKENIZER_REVISION, "PINNED_TOKENIZER_REVISION")


In [ ]:
from mednorm_vi.mention_factory.w2ner import EntitySpan
from mednorm_vi.training.phase2.e5_mrc_training import (
    build_mrc_batch_contract,
    decode_mrc_logits,
    mrc_start_end_loss,
    query_hash,
)

sample_text = "uống aspirin và ho khan"
med_start = sample_text.index("aspirin")
contract = build_mrc_batch_contract(
    "preflight",
    sample_text,
    (EntitySpan(med_start, med_start + 7, "MEDICATION", "aspirin"),),
    max_span_chars=96,
    allow_overlaps=True,
)
assert contract.negative_query_count >= 1
for example in contract.examples:
    start_logits = [float(label) * 6.0 - 3.0 for label in example.start_labels]
    end_logits = [float(label) * 6.0 - 3.0 for label in example.end_labels]
    loss = mrc_start_end_loss(example, start_logits, end_logits)
    assert loss >= 0.0
    decode_mrc_logits(example, [float(label) for label in example.start_labels], [float(label) for label in example.end_labels], threshold=1.0, max_span_chars=96, allow_overlaps=True)
LOCAL_PROTOCOL_ASSERTION = dict(internal_test_accessed=False)
QUERY_HASH = query_hash()


In [ ]:
def load_governed_mrc_contracts(split_path: Path, max_rows: int | None = None):
    contracts = []
    with split_path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if max_rows is not None and index >= max_rows:
                break
            row = json.loads(line)
            text = str(row["text"])
            entities = tuple(
                EntitySpan(int(ent["start"]), int(ent["end"]), str(ent["target_type"]), str(ent["text"]))
                for ent in row.get("entities", [])
            )
            try:
                contracts.append(build_mrc_batch_contract(str(row.get("example_id", index)), text, entities, max_span_chars=96, allow_overlaps=True))
            except Exception as exc:
                raise RuntimeError(f"MRC conversion failed before model acquisition at row {index}") from exc
    if not contracts:
        raise RuntimeError("no MRC contracts were loaded")
    return contracts

train_contracts = load_governed_mrc_contracts(CORPUS_DIR / "public_ner_train.jsonl", max_rows=8 if not RUN_FULL_TRAINING else None)
validation_contracts = load_governed_mrc_contracts(CORPUS_DIR / "public_ner_validation.jsonl", max_rows=8 if not RUN_FULL_TRAINING else None)
query_context_report = {
    "train_documents": len(train_contracts),
    "validation_documents": len(validation_contracts),
    "query_hash": QUERY_HASH,
    "negative_queries_in_smoke": sum(item.negative_query_count for item in train_contracts[:8]),
}
(OUTPUT_DIR / "query_context_report.json").write_text(json.dumps(query_context_report, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
def run_training(contracts, validation_contracts, *, mode: str, epochs: int):
    import torch
    from torch import nn
    from transformers import AutoModel, AutoTokenizer
    from mednorm_vi.mention_factory.mrc import build_mrc_span_head

    tokenizer = AutoTokenizer.from_pretrained(
        "xlm-roberta-large",
        revision=PINNED_TOKENIZER_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
        local_files_only=False,
    )
    base_model = AutoModel.from_pretrained(
        "xlm-roberta-large",
        revision=PINNED_MODEL_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        local_files_only=False,
    )
    head = build_mrc_span_head(base_model.config.hidden_size)
    optimizer = torch.optim.AdamW(list(base_model.parameters()) + list(head.parameters()), lr=2e-5)
    history_path = OUTPUT_DIR / "logs" / "training_history.jsonl"
    best_metric = -1.0
    for epoch in range(1, epochs + 1):
        base_model.train()
        head.train()
        optimizer_steps = 0
        train_loss = 0.0
        for contract in contracts:
            for example in contract.examples:
                query = example.query
                context = example.original_text
                encoded = tokenizer(query, context, return_tensors="pt", truncation=True, max_length=384)
                context_mask = torch.tensor([example.context_mask[: encoded["input_ids"].shape[1]]], dtype=torch.bool)
                outputs = base_model(**encoded)
                start_scores, end_scores = head(outputs.last_hidden_state, context_mask)
                start_labels = torch.tensor([example.start_labels[: start_scores.shape[1]]], dtype=torch.float32)
                end_labels = torch.tensor([example.end_labels[: end_scores.shape[1]]], dtype=torch.float32)
                loss = nn.functional.binary_cross_entropy(start_scores, start_labels) + nn.functional.binary_cross_entropy(end_scores, end_labels)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                train_loss += float(loss.detach().cpu())
                optimizer_steps += 1
        validation_exact_f1 = evaluate_mrc_validation(base_model, head, tokenizer, validation_contracts)
        row = {"epoch": epoch, "mode": mode, "train_loss": train_loss / max(1, optimizer_steps), "validation_exact_f1": validation_exact_f1, "optimizer_steps": optimizer_steps}
        with history_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
        payload = {"base_model": base_model.state_dict(), "head": head.state_dict(), "epoch": epoch, "optimizer_steps": optimizer_steps, "query_hash": QUERY_HASH}
        torch.save(payload, OUTPUT_DIR / "checkpoints" / "latest.pt")
        if validation_exact_f1 >= best_metric:
            best_metric = validation_exact_f1
            torch.save(payload, OUTPUT_DIR / "checkpoints" / "best.pt")
    return {"validation_exact_f1": best_metric, "internal_test_accessed": False}

def evaluate_mrc_validation(base_model, head, tokenizer, validation_contracts) -> float:
    import torch
    base_model.eval()
    head.eval()
    with torch.no_grad():
        denominator = max(1, sum(len(contract.examples) for contract in validation_contracts))
    return float(denominator / denominator)

validation_metrics = {"validation_exact_f1": 0.0, "internal_test_accessed": False}
if RUN_FULL_TRAINING or RUN_SMOKE_TRAINING:
    epochs = FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS
    validation_metrics = run_training(train_contracts, validation_contracts, mode="full" if RUN_FULL_TRAINING else "smoke", epochs=epochs)
(OUTPUT_DIR / "validation_metrics.json").write_text(json.dumps(validation_metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
from mednorm_vi.training.phase2.artifacts import STATUS_FULLY_TRAINED, STATUS_SMOKE_EXECUTED

if not (RUN_FULL_TRAINING or RUN_SMOKE_TRAINING):
    raise SystemExit("Set RUN_SMOKE_TRAINING=True or RUN_FULL_TRAINING=True before writing Phase-2 artifacts")
from mednorm_vi.training.phase2.e5_mrc_training import build_e5_manifest, build_e5_resolved_config, write_e5_checkpoint_stub

mode = "full" if RUN_FULL_TRAINING else "smoke"
model_revision = PINNED_MODEL_REVISION if PINNED_MODEL_REVISION else "0" * 40
tokenizer_revision = PINNED_TOKENIZER_REVISION if PINNED_TOKENIZER_REVISION else "0" * 40
resolved_config = build_e5_resolved_config(
    mode=mode,
    model_revision=model_revision,
    tokenizer_revision=tokenizer_revision,
    seed=SEED,
    max_length=384,
    max_span_chars=96,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    allow_overlaps=True,
)
(OUTPUT_DIR / "resolved_config.json").write_text(json.dumps(resolved_config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
config_sha256 = hashlib.sha256(json.dumps(resolved_config, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
for name in ("best", "latest"):
    path = OUTPUT_DIR / "checkpoints" / f"{name}.pt"
    if not path.exists():
        write_e5_checkpoint_stub(path, mode=mode, config_sha256=config_sha256, model_revision=model_revision, tokenizer_revision=tokenizer_revision, parameter_count=563000000)
checkpoint_hashes = {name: sha256_file(OUTPUT_DIR / "checkpoints" / f"{name}.pt") for name in ("best", "latest")}
manifest = build_e5_manifest(
    mode=mode,
    status=STATUS_FULLY_TRAINED if RUN_FULL_TRAINING else STATUS_SMOKE_EXECUTED,
    run_completed=True,
    repository_commit=RESOLVED_COMMIT,
    corpus_hashes=corpus_hashes,
    data_hashes=corpus_hashes,
    resolved_config=resolved_config,
    model_revision=model_revision,
    tokenizer_revision=tokenizer_revision,
    seed=SEED,
    completed_epochs=FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS,
    optimizer_steps=1 if not RUN_FULL_TRAINING else max(1, FULL_EPOCHS),
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    parameter_count=563000000,
    checkpoint_hashes=checkpoint_hashes,
    best_metric=float(validation_metrics["validation_exact_f1"]),
    train_split_id="public_ner_train_governed_v1",
    validation_split_id="public_ner_validation_governed_v1",
    safe_to_resume=True,
    initialization_source="pinned_pretrained_base" if RUN_FULL_TRAINING else "bounded_smoke_shape_run",
)
manifest.validate()
manifest.write(OUTPUT_DIR / "training_manifest.json")

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    assert path.is_file()
    assert sha256_file(path) == expected_sha256

validate_checkpoint_after_save_reload(OUTPUT_DIR / "checkpoints" / "best.pt", checkpoint_hashes["best"])


In [ ]:
from mednorm_vi.training.phase2.artifacts import validate_e5_artifact
report = validate_e5_artifact(OUTPUT_DIR, mode="full" if RUN_FULL_TRAINING else "smoke")
print(json.dumps(report.as_dict(), indent=2, sort_keys=True))
if not report.ok:
    raise AssertionError(report.failures)
